# RAG — Retrieval-Augmented Generation

RAG is the practice of giving a language model access to an external knowledge base at inference time.  
Instead of relying purely on what the model memorised during training, it retrieves relevant passages and uses them as context.

**This notebook:**
1. Builds a medical corpus from Wikipedia (25 topics, chunked into passages)
2. Indexes it into ChromaDB using sentence-transformers embeddings
3. Defines a DSPy RAG module: retrieve → generate
4. Measures baseline accuracy (no retrieval) vs RAG accuracy
5. Optimizes the RAG pipeline with MIPROv2

In [1]:
import re
import random
import warnings
import numpy as np
warnings.filterwarnings('ignore')

import dspy
import wikipedia
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from dotenv import load_dotenv
load_dotenv()

lm = dspy.LM('openai/gpt-4o-mini')
dspy.configure(lm=lm)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Ready.')

Ready.


## Step 1 — Build the medical corpus

We fetch 25 Wikipedia articles on medical conditions and drugs, then chunk each article into ~150-word passages.  
These passages become the knowledge base the retriever will search over.

In [2]:
MEDICAL_TOPICS = [
    "Diabetes mellitus", "Hypertension", "Asthma", "Myocardial infarction",
    "Stroke", "Pneumonia", "Tuberculosis", "Malaria", "HIV/AIDS",
    "Alzheimer's disease", "Parkinson's disease", "Epilepsy",
    "Chronic kidney disease", "Liver cirrhosis", "Anemia",
    "Hypothyroidism", "Rheumatoid arthritis", "Osteoporosis",
    "Metformin", "Aspirin", "Amoxicillin", "Atorvastatin",
    "Paracetamol", "Warfarin", "Insulin"
]

def chunk_text(text, chunk_size=150):
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

passages = []   # list of dicts: {id, text, topic}
wikipedia.set_lang('en')

for topic in MEDICAL_TOPICS:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        chunks = chunk_text(page.content)
        for i, chunk in enumerate(chunks):
            passages.append({'id': f'{topic}_{i}', 'text': chunk, 'topic': topic})
    except Exception as e:
        print(f'Skipped {topic}: {e}')

print(f'Total passages indexed: {len(passages)}')
print(f'Sample: {passages[0]["text"][:200]}')

Skipped Diabetes mellitus: Expecting value: line 1 column 1 (char 0)
Skipped Hypertension: Expecting value: line 1 column 1 (char 0)
Skipped Asthma: Expecting value: line 1 column 1 (char 0)
Skipped Myocardial infarction: Expecting value: line 1 column 1 (char 0)
Skipped Stroke: Expecting value: line 1 column 1 (char 0)
Skipped Pneumonia: Expecting value: line 1 column 1 (char 0)
Skipped Tuberculosis: Expecting value: line 1 column 1 (char 0)
Skipped Malaria: Expecting value: line 1 column 1 (char 0)
Skipped HIV/AIDS: Expecting value: line 1 column 1 (char 0)
Skipped Alzheimer's disease: Expecting value: line 1 column 1 (char 0)
Skipped Parkinson's disease: Expecting value: line 1 column 1 (char 0)
Total passages indexed: 583
Sample: Epilepsy is a group of neurological disorders characterized by a tendency for recurrent, unprovoked seizures. A seizure is a sudden burst of abnormal electrical activity in the brain that can cause a 


## Step 2 — Build an in-memory vector index

We embed every passage with `all-MiniLM-L6-v2` and store the embeddings in a numpy matrix.  
At query time the question is embedded and cosine similarity is computed against all passages — the top-k are returned.  
No external vector DB needed for a corpus this size.

In [3]:
corpus_texts = [p['text'] for p in passages]

# Embed and L2-normalise so dot product == cosine similarity
corpus_embeddings = embedder.encode(corpus_texts, show_progress_bar=True)
corpus_embeddings /= np.linalg.norm(corpus_embeddings, axis=1, keepdims=True)

print(f'Indexed {len(corpus_texts)} passages.')

def retrieve(query: str, k: int = 3) -> list[str]:
    q_emb = embedder.encode([query])
    q_emb /= np.linalg.norm(q_emb)
    scores = corpus_embeddings @ q_emb.T          # cosine similarity for all passages
    top_k  = np.argsort(scores.flatten())[-k:][::-1]
    return [corpus_texts[i] for i in top_k]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Indexed 583 passages.


## Step 3 — Load QA dataset

We use `medmcqa` filtered to **Pharmacology** questions — these are best covered by our drug articles.  
The correct option is converted from a multiple-choice index to the actual answer text.

In [4]:
raw = load_dataset('medmcqa', split='validation')

OPTIONS = ['opa', 'opb', 'opc', 'opd']

examples = [
    dspy.Example(
        question=row['question'],
        answer=row[OPTIONS[row['cop']]]
    ).with_inputs('question')
    for row in raw
    if row['subject_name'] == 'Pharmacology'
    and row['cop'] is not None
    and all(row[o] for o in OPTIONS)
]

random.seed(42)
random.shuffle(examples)
trainset = examples[:150]
devset   = examples[150:250]

print(f'Pharmacology examples: {len(examples)}')
print(f'Train: {len(trainset)} | Dev: {len(devset)}')
print(f'\nSample Q: {devset[0].question}')
print(f'Sample A: {devset[0].answer}')

Pharmacology examples: 243
Train: 150 | Dev: 93

Sample Q: Pegloticase used in which of the following conditions?
Sample A: Chronic Gout


## Step 4 — Define the RAG module

The pipeline has two steps:
1. **Retrieve** — call ChromaDB to get the top-3 most relevant passages
2. **Generate** — pass the question + passages to the LM and produce an answer

Both steps are wired together in `forward()`.

In [5]:
class GenerateAnswer(dspy.Signature):
    """Answer the medical question using only the provided context passages."""

    context:  list[str] = dspy.InputField(desc="Relevant passages from the medical knowledge base")
    question: str       = dspy.InputField()
    answer:   str       = dspy.OutputField(desc="Concise answer, a few words or a short phrase")


class MedicalRAG(dspy.Module):
    def __init__(self, k=3):
        self.k = k
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        context = retrieve(question, k=self.k)
        return self.generate(context=context, question=question)

## Step 5 — Write the metric

Exact string match is too strict for free-text answers (capitalisation, short forms).  
We do a case-insensitive substring check: the correct answer must appear somewhere in the prediction.

In [6]:
def answer_match(example, prediction, trace=None):
    expected  = example.answer.lower().strip()
    predicted = prediction.answer.lower().strip()
    return float(expected in predicted or predicted in expected)

## Step 6 — Baseline: LM only (no retrieval)

First we measure how well gpt-4o-mini answers pharmacology questions from memory alone, with no context.

In [7]:
class AnswerOnly(dspy.Signature):
    """Answer the medical question."""
    question: str = dspy.InputField()
    answer:   str = dspy.OutputField(desc="Concise answer, a few words or a short phrase")

class NoRAG(dspy.Module):
    def __init__(self):
        self.generate = dspy.ChainOfThought(AnswerOnly)

    def forward(self, question):
        return self.generate(question=question)

evaluate = dspy.evaluate.Evaluate(devset=devset, metric=answer_match, num_threads=4, display_progress=True)

no_rag_score = evaluate(NoRAG())
print(f'LM only (no retrieval): {no_rag_score.score:.1f}%')

Average Metric: 22.00 / 93 (23.7%): 100%|██████████| 93/93 [00:01<00:00, 56.39it/s] 

2026/09/03 15:17:11 INFO dspy.evaluate.evaluate: Average Metric: 22.0 / 93 (23.7%)



LM only (no retrieval): 23.7%


## Step 7 — RAG baseline

Same evaluation, but now the model receives the top-3 retrieved passages as context.

In [8]:
rag_baseline = MedicalRAG(k=3)
rag_score = evaluate(rag_baseline)
print(f'LM only : {no_rag_score.score:.1f}%')
print(f'RAG     : {rag_score.score:.1f}%')
print(f'Delta   : +{rag_score.score - no_rag_score.score:.1f}%')

  0%|          | 0/93 [00:00<?, ?it/s]

Average Metric: 6.00 / 93 (6.5%): 100%|██████████| 93/93 [00:21<00:00,  4.32it/s]

2026/09/03 15:17:38 INFO dspy.evaluate.evaluate: Average Metric: 6.0 / 93 (6.5%)



LM only : 23.7%
RAG     : 6.5%
Delta   : +-17.2%


## Step 8 — Results: LM only vs RAG

Two numbers side by side — this is the core RAG lesson.
If retrieval is working, the gap should be clear.

> **Note:** MIPROv2 optimization is skipped here due to an optuna/numpy incompatibility in this environment.
The full optimization workflow is already demonstrated in `05_mipro.ipynb`.

In [10]:
print(f'LM only (no RAG)  : {no_rag_score.score:.1f}%')
print(f'RAG (unoptimized) : {rag_score.score:.1f}%')
print(f'Delta             : +{rag_score.score - no_rag_score.score:.1f}%')

LM only (no RAG)  : 23.7%
RAG (unoptimized) : 6.5%
Delta             : +-17.2%


## Step 9 — Inspect a live retrieval

In [11]:
# Run a live query to see what passages get retrieved
q = "What is the mechanism of action of Metformin?"
retrieved = retrieve(q, k=3)
print("Retrieved passages:\n")
for i, p in enumerate(retrieved):
    print(f"[{i+1}] {p[:250]}...\n")

result = rag_baseline(question=q)
print(f"Answer: {result.answer}")

Retrieved passages:

[1] the liver. Metformin has indirect antiandrogenic effects in women with insulin resistance, such as those with PCOS, due to its beneficial effects on insulin sensitivity. It may reduce testosterone levels in such women by as much as 50%. A Cochrane re...

[2] Metformin, sold under the brand name Glucophage, among others, is the main first-line medication for the treatment of type 2 diabetes, particularly in people who are overweight. It is also used in the treatment of polyendocrine metabolic ovarian synd...

[3] the kidneys; both metformin and cimetidine are cleared from the body by tubular secretion, and both, particularly the cationic (positively charged) form of cimetidine, may compete for the same transport mechanism. A small double-blind, randomized stu...

Answer: Inhibits liver glucose production and increases insulin sensitivity.
